# Strands Agents와 AgentCore Memory(단기 메모리)


## 소개

이 튜토리얼에서는 Strands Agents와 AgentCore **단기 메모리**(Raw event)를 사용하여 **Personal Agent**를 구축하는 방법을 살펴봅니다. Agent는 `get_last_k_turns`를 사용하여 세션의 최근 대화를 기억하며, 사용자가 돌아오면 대화를 자연스럽게 이어 갈 수 있습니다.


### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화                                                        |
| Agent 유형          | Personal Agent                                                                   |
| Agentic Framework   | Strands Agents                                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, AgentInitializedEvent and MessageAddedEvent hooks   |
| 예제 난이도  | 초급                                                                         |

다음 내용을 학습합니다.
- 대화 연속성을 위한 단기 메모리 사용
- 최근 K개의 대화 turn 검색
- 실시간 정보를 위한 웹 검색 도구 사용
- 대화 기록으로 Agent 초기화

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항

- Python 3.10 이상
- AgentCore Memory 권한이 있는 AWS 자격 증명
- AgentCore Memory 역할 ARN
- Amazon Bedrock 모델에 대한 액세스

먼저 환경을 설정하겠습니다.

## 1단계: 설정 및 가져오기

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime

# 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("personal-agent")

In [ ]:
# 가져오기
import os
from strands import Agent, tool
from strands.hooks import (
    AgentInitializedEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)
from bedrock_agentcore.memory import MemoryClient

# 구성
REGION = os.getenv("AWS_REGION", "us-west-2")  # Agent용 AWS 리전
ACTOR_ID = "user_123"  # 고유 식별자라면 무엇이든 사용 가능(AgentID, User ID 등)
SESSION_ID = "personal_session_001"  # 고유 세션 식별자

## 2단계: 웹 검색 도구

먼저 Agent에서 사용할 간단한 웹 검색 도구를 생성합니다.

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS


@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.

    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.

    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"


logger.info("✅ Web search tool ready")

## 3단계: Memory 리소스 생성
단기 메모리에서는 strategy 없이 Memory 리소스를 생성합니다. 여기에는 `get_last_k_turns`로 검색할 수 있는 원시 대화 turn이 저장됩니다.


In [ ]:
from botocore.exceptions import ClientError

# Memory Client 초기화
client = MemoryClient(region_name=REGION)
memory_name = "PersonalAgentMemory"

try:
    # strategy 없이 Memory 리소스 생성(단기 메모리만 사용)
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 strategy를 사용하지 않음
        description="Short-term memory for personal agent",
        event_expiry_days=7,  # 단기 메모리 보존 기간. 최대 365일까지 설정 가능
    )
    memory_id = memory["id"]
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 표시
    logger.error(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

## 4단계: Memory Hook

이 단계에서는 메모리 작업을 자동화하는 사용자 지정 `MemoryHookProvider` 클래스를 정의합니다. Hook은 Agent 실행 수명 주기의 특정 시점에 실행되는 특수 함수입니다. 여기서 만드는 Memory Hook에는 두 가지 주요 기능이 있습니다.
1. **최근 대화 불러오기**: `AgentInitializedEvent` Hook을 사용하여 Agent 초기화 시 최근 대화 기록을 자동으로 불러옵니다.
2. **마지막 메시지 저장**: 새 대화 메시지를 저장합니다.

이를 통해 수동 관리 없이 자연스러운 메모리 환경을 구현할 수 있습니다.

In [ ]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_client: MemoryClient, memory_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        try:
            # Agent 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Memory에서 최근 5개의 대화 turn 불러오기
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=self.memory_id, actor_id=actor_id, session_id=session_id, k=5
            )

            if recent_turns:
                # 대화 기록을 컨텍스트 형식으로 변환
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message["role"]
                        content = message["content"]["text"]
                        context_messages.append(f"{role}: {content}")

                context = "\n".join(context_messages)
                # Agent의 system prompt에 컨텍스트 추가
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns")

        except Exception as e:
            logger.error(f"Memory load error: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """메시지를 메모리에 저장합니다."""
        messages = event.agent.messages
        try:
            # Agent 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if messages[-1]["content"][0].get("text"):
                self.memory_client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=[(messages[-1]["content"][0]["text"], messages[-1]["role"])],
                )
        except Exception as e:
            logger.error(f"Memory save error: {e}")

    def register_hooks(self, registry: HookRegistry):
        # Memory Hook 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 5단계: 웹 검색 기능을 갖춘 Personal Agent 생성

In [ ]:
def create_personal_agent():
    """메모리와 웹 검색 기능이 있는 개인 에이전트를 생성합니다."""
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        system_prompt=f"""You are a helpful personal assistant with web search capabilities.
        
        You can help with:
        - General questions and information lookup
        - Web searches for current information
        - Personal task management
        
        When you need current information, use the websearch function.
        Today's date: {datetime.today().strftime("%Y-%m-%d")}
        Be friendly and professional.""",
        hooks=[MemoryHookProvider(client, memory_id)],
        tools=[websearch],
        state={"actor_id": ACTOR_ID, "session_id": SESSION_ID},
    )
    return agent


# Agent 생성
agent = create_personal_agent()
logger.info("✅ Personal agent created with memory and web search")

#### 축하합니다! Agent가 준비되었습니다! :)
## Agent 테스트

In [ ]:
# 메모리가 적용된 대화 테스트
print("=== First Conversation ===")
print("User: My name is Alex and I'm interested in learning about AI.")
print("Agent: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

In [ ]:
print("User: Can you search for the latest AI trends in 2025?")
print("Agent: ", end="")
agent("Can you search for the latest AI trends in 2025?")

In [ ]:
print("User: I'm particularly interested in machine learning applications.")
print("Agent: ", end="")
agent("I'm particularly interested in machine learning applications.")

## 메모리 연속성 테스트

메모리 시스템이 올바르게 작동하는지 확인하기 위해 새 Agent 인스턴스를 생성하고 이전에 저장한 정보에 액세스할 수 있는지 살펴보겠습니다.

In [ ]:
# 새 Agent 인스턴스 생성(사용자가 돌아오는 상황 모의)
print("=== User Returns - New Session ===")
new_agent = create_personal_agent()

# 메모리 연속성 테스트
print("User: What was my name again?")
print("Agent: ", end="")
new_agent("What was my name again?")

print("User: Can you search for more information about machine learning?")
print("Agent: ", end="")
new_agent("Can you search for more information about machine learning?")

## 저장된 메모리 확인

In [ ]:
# Memory에 저장된 내용 확인
print("=== Memory Contents ===")
recent_turns = client.get_last_k_turns(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
    k=3,  # 더 많거나 적은 turn을 보려면 k 조정
)

for i, turn in enumerate(recent_turns, 1):
    print(f"Turn {i}:")
    for message in turn:
        role = message["role"]
        content = (
            message["content"]["text"][:100] + "..."
            if len(message["content"]["text"]) > 100
            else message["content"]["text"]
        )
        print(f"  {role}: {content}")
    print()

## 요약

이 튜토리얼에서는 Personal Agent를 구축했습니다. 학습한 내용은 다음과 같습니다.

- strategy 없이 Memory 리소스 생성
- 대화 기록에 `get_last_k_turns` 사용
- Agent에 웹 검색 기능 추가
- 컨텍스트를 불러오기 위한 Memory Hook 구현

**다음 단계:**
- 더 정교한 도구 추가
- 장기 메모리 strategy 구현
- 여러 소스로 검색 기능 강화

## 리소스 정리(선택 사항)

In [ ]:
# Memory 리소스를 삭제하려면 주석 해제
# client.delete_memory_and_wait(memory_id)
# logger.info(f"✅ Deleted memory: {memory_id}")